# 🧪 Expérimentation MLOps (Docker, MLflow, MinIO & Discord)
Ce notebook démontre comment s'intègre toute l'architecture.

In [8]:
import sys
import os
import boto3
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Ajout du chemin parent pour importer src
sys.path.append(os.path.abspath('../'))
from src.utils.discord_notify import notify_discord

# 1. Création automatique du Bucket S3 (MinIO) si inexistant
s3 = boto3.client('s3', 
                  endpoint_url='http://minio:9000', 
                  aws_access_key_id='root', 
                  aws_secret_access_key='root123456789')
try:
    s3.create_bucket(Bucket='mlflow')
    print("Bucket 'mlflow' créé dans MinIO.")
except Exception:
    print("Bucket 'mlflow' déjà existant.")

# 2. Connexion au Tracking Server MLflow (Conteneur Docker)
mlflow.set_tracking_uri('http://mlflow:5000')


Bucket 'mlflow' déjà existant.


In [9]:
notify_discord("Début d'une session Jupyter dans l'environnement Dockerisé...", "info")

try:
    # Chargement des données
    data = load_breast_cancer()
    X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2)
    
    mlflow.set_experiment("Classification_Cancer")
    mlflow.sklearn.autolog()
    
    # Entraînement
    
    with mlflow.start_run() as run:
        notify_discord(f"Entraînement du modèle (Run ID: `{run.info.run_id}`)...", "info")
        
        clf = RandomForestClassifier(n_estimators=100, max_depth=5)
        clf.fit(X_train, y_train)
        
        # Evaluation
        preds = clf.predict(X_test)
        acc = accuracy_score(y_test, preds)
        
        notify_discord(f"Entraînement terminé avec succès !\n**Précision :** `{acc:.4f}`\n*Le modèle a été sauvegardé dans MinIO via MLflow.*", "success")

except Exception as e:
    notify_discord(f"Alerte lors de l'entraînement :\n```python\n{str(e)}\n```", "error")
    raise e

2026/05/11 20:26:15 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.5.0 <= scikit-learn, but the installed version is 1.3.1. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/05/11 20:26:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run bold-wolf-897 at: http://mlflow:5000/#/experiments/1/runs/3fd03fe2a24548e1b630414cefd61f55
🧪 View experiment at: http://mlflow:5000/#/experiments/1


In [ ]:
import requests
import json
from datetime import datetime
import logging
import os

# Configuration du logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("DiscordNotifier")

# Il est préférable d'utiliser une variable d'environnement pour la sécurité
WEBHOOK_URL = os.getenv("DISCORD_WEBHOOK_URL", "https://discord.com/api/webhooks/1502639151034404999/wP9lUn1qgL7QCVeaLp-2OwgNYjc8oXddz7Z2b5z2IQaaOhZGm-zKPbN2qhhHGvuV74LG")

#WEBHOOK_URL = "https://discord.com/api/webhooks/1502639151034404999/wP9lUn1qgL7QCVeaLp-2OwgNYjc8oXddz7Z2b5z2IQaaOhZGm-zKPbN2qhhHGvuV74LG"

#def notify_discord(message, level="info"):
#def notify_discord(message, level="info", task_name="Training"):
"""def notify_discord(message, metrics=None, level="success",task_name="Training"):
    " ""
    Envoie une notification enrichie sur Discord.
    level: 'info', 'success', 'error', 'warning'
    "" "
    colors = {
        "info": 3447003,      # Bleu
        "success": 3066993,   # Vert
        "warning": 16776960,  # Jaune
        "error": 15158332     # Rouge
    }
    # Transformation des métriques en texte lisible
    metrics_text = "N/A"
    if metrics:
        metrics_text = "\n".join([f"**{k}:** `{v:.4f}`" for k, v in metrics.items()])

    emoji = {"info": "ℹ️", "success": "✅", "error": "❌", "warning": "⚠️"}
    #emoji = {"info": "ℹ️", "success": "✅", "error": "❌"}
    
    payload = {
        "username": "MLOps Watcher",
        "avatar_url": "https://cdn-icons-png.flaticon.com/512/2103/2103633.png",
        "embeds": [{
            "title": f"{emoji.get(level, '➡️')} {task_name.upper()}",
            "description": message,
            "color": colors.get(level, 3447003),
            "fields": [
                {
                    "name": "Environnement-Model",
                    "value": "`Production/Lab`",
                    "inline": True
                },
                {
                    "name": "Auteur",
                    "value": "`ING.LABIADH LINA`",
                    "inline": True
                },
                {
                    "name": "Horodatage",
                    "value": datetime.now().strftime("%H:%M:%S"),
                    "inline": True
                },
                
                {
                    "name": "📊 Métriques",
                    "value": metrics_text,
                    "inline": False
                },
                {
                    "name": "🔗 MLflow UI",
                    "value": "[Accéder à l'interface](http://localhost:5000)",
                    "inline": False
                }
            ],
            "thumbnail": {
                "url": "https://cdn-icons-png.flaticon.com/512/4712/4712139.png" # Icône de fusée/IA
            },
            "footer": {
                "text": "Système de Monitoring MLOps - iTeam Univ"
            },
            "timestamp": datetime.utcnow().isoformat()
        }]
    }
"""
    
def notify_discord(message, run_id="N/A", metrics=None, level="success", task_name="Training"):
    """
    Notification Discord Optimisée - Style iTeam Univ
    """
    colors = {
        "info": 3447003, "success": 3066993, "warning": 16776960, "error": 15158332
    }
    
    metrics_text = "N/A"
    if metrics:
        metrics_text = "\n".join([f"**{k}:** `{v:.4f}`" for k, v in metrics.items()])

    emoji = {"info": "ℹ️", "success": "✅", "error": "❌", "warning": "⚠️"}
    
    payload = {
        "username": "MLOps Watcher",
        "avatar_url": "https://cdn-icons-png.flaticon.com/512/2103/2103633.png",
        "embeds": [{
            "title": f"{emoji.get(level, '➡️')} {task_name.upper()}",
            "description": message,
            "color": colors.get(level, 3447003),
            "fields": [
                {"name": "🌍 Environnement", "value": "`Production/Lab`", "inline": True},
                {"name": "👤 Auteur", "value": "`ING. LABIADH LINA`", "inline": True},
                {"name": "🆔 Run ID", "value": f"`{run_id}`", "inline": True},
                {"name": "📊 Métriques", "value": metrics_text, "inline": False},
                {"name": "🔗 MLflow UI", "value": "[Accéder à l'interface](http://localhost:5000)", "inline": False}
            ],
            "thumbnail": {"url": "https://cdn-icons-png.flaticon.com/512/4712/4712139.png"},
            "footer": {"text": "Système de Monitoring MLOps - iTeam Univ"},
            "timestamp": datetime.utcnow().isoformat()
        }]
    }
    
    try:
        response = requests.post(
            WEBHOOK_URL, 
            data=json.dumps(payload), 
            headers={'Content-Type': 'application/json'},
            timeout=5
        )
        response.raise_for_status()
        logger.info(f"Notification Discord envoyée avec succès : {level}")
    except requests.exceptions.HTTPError as err:
        logger.error(f"Erreur HTTP Discord : {err}")
    except Exception as e:
        logger.error(f"Erreur inattendue lors de l'envoi Discord : {e}")

"""if __name__ == "__main__":
    # Test rapide
    notify_discord("Pipeline MLOps démarré avec succès sur Docker.",
                   metrics={"R2_Score": 0.8942},
                   level="success",
                   task_name="Training"
                   )"""

In [7]:
notify_discord("Pipeline MLOps démarré avec succès sur Docker.",metrics={"R2_Score": 0.8942},level="success",task_name="Training")

try:
    # Chargement des données
    data = load_breast_cancer()
    X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2)
    
    mlflow.set_experiment("Classification_Cancer")
    mlflow.sklearn.autolog()
    
    with mlflow.start_run() as run:
        # On précise level="info" explicitement
        notify_discord(f"Début de l'entraînement...", level="info", task_name="ML-Pipeline")
        
        clf = RandomForestClassifier(n_estimators=100, max_depth=5)
        clf.fit(X_train, y_train)
        
        # Evaluation
        preds = clf.predict(X_test)
        acc = accuracy_score(y_test, preds)
        
        # On passe les métriques dans le dictionnaire dédié
        notify_discord(
            message="Le modèle a été sauvegardé dans MinIO via MLflow.", 
            metrics={"Accuracy": acc}, 
            level="success",
            task_name="Training-Complete"
        )

except Exception as e:
    # En cas d'erreur, on utilise le level "error"
    notify_discord(f"Erreur critique détectée :\n{str(e)}", level="error", task_name="Failure")
    raise e

TypeError: notify_discord() got an unexpected keyword argument 'metrics'